# COD-22: policy-priority matrix review

This notebook rebuilds the type-level policy candidates without rewriting tracked artifacts. Actions remain candidates for field validation.

In [ ]:
import os
from pathlib import Path

current = Path.cwd().resolve()
project_root = next(
    path for path in (current, *current.parents)
    if (path / 'pyproject.toml').is_file()
)
os.chdir(project_root)
project_root

In [ ]:
import json

import pandas as pd
import plotly.express as px

from busan_imd.policy_matrix import (
    DEFAULT_ASSIGNMENT_OUTPUT,
    DEFAULT_CATALOG,
    DEFAULT_CLUSTER_REPORT,
    DEFAULT_OVERLAY,
    build,
)

assignments = pd.read_csv(
    DEFAULT_ASSIGNMENT_OUTPUT, dtype={'admin_dong_code': str}
)
overlay = pd.read_csv(DEFAULT_OVERLAY, dtype={'admin_dong_code': str})
catalog = pd.read_csv(DEFAULT_CATALOG)
cluster_report = json.loads(
    DEFAULT_CLUSTER_REPORT.read_text(encoding='utf-8')
)
matrix, report = build(assignments, overlay, catalog, cluster_report)
report['cluster_summaries']

In [ ]:
domain_actions = matrix[matrix['policy_trigger'].str.startswith('domain:')].copy()
evidence = px.bar(
    domain_actions,
    x='analysis_basis_value',
    y='policy_title_ko',
    color='cluster_label',
    facet_row='cluster_label',
    orientation='h',
    text='analysis_basis_value',
    hover_data=['target_area_count', 'lead_implementer'],
    title='Positive mean domain-excess evidence behind policy candidates',
)
evidence.update_yaxes(matches=None)
evidence.show()

In [ ]:
targets = px.scatter(
    matrix,
    x='cluster_label',
    y='policy_title_ko',
    size='target_area_count',
    color='implementation_difficulty',
    symbol='policy_trigger',
    hover_data=[
        'target_admin_dongs', 'lead_implementer', 'implementation_partners',
        'expected_effect', 'monitoring_indicator', 'evidence_limit',
    ],
    title='Policy candidates, target size, and implementation difficulty',
)
targets.update_traces(marker={'sizemin': 12})
targets.show()

In [ ]:
matrix[[
    'cluster_label', 'policy_priority', 'policy_title_ko',
    'target_area_count', 'target_admin_dongs', 'analysis_basis',
    'analysis_basis_value', 'lead_implementer',
    'implementation_difficulty', 'expected_effect',
    'monitoring_indicator', 'decision_status',
]]